# Loading geomstats data from DHOOM files

[DHOOM](https://dhoom.dev) — *Davis Human-readable Optimized Object Markup* — is a compact, human-readable serialization format for structured data. It encodes the same data model as JSON, but exploits the **fiber-bundle decomposition** of homogeneous collections to factor out structural redundancy: the schema is declared once in the header, then records carry only what deviates from it.

Geomstats supports DHOOM via two functions, parallel to `json.load` and `json.dump`:

- `geomstats.datasets.utils.load_dhoom(path)`
- `geomstats.datasets.utils.save_dhoom(value, path)`

This notebook walks through:

1. Loading the cities dataset from both JSON and DHOOM and confirming they produce the same array.
2. Comparing the on-disk size of the two formats.
3. Reading the DHOOM file to see the fiber structure for itself.
4. Round-tripping a small dataset through `save_dhoom`.

## 1. Semantic equivalence with `load_cities`

`load_cities()` reads `cities.json` and applies a lat/lng-to-spherical-extrinsic transform, returning a `(50, 3)` array of points on the 2-sphere plus a list of city names. We can reproduce that exactly from the DHOOM file by calling `load_dhoom` and applying the same transform.

In [1]:
import os

import geomstats.backend as gs
import geomstats.datasets.utils as data_utils
from geomstats.geometry.hypersphere import Hypersphere

sphere = Hypersphere(dim=2)

# Existing JSON loader — does the spherical transform inside.
cities_from_json, names = data_utils.load_cities()

# Generic DHOOM loader — returns the raw list of records.
raw = data_utils.load_dhoom(data_utils.CITIES_DHOOM_PATH)

# Apply the same transform load_cities applies internally.
lat_lng = gs.array(
    [[row['lat'] / 180 * gs.pi, row['lng'] / 180 * gs.pi] for row in raw]
)
colat = gs.expand_dims(gs.pi / 2 - lat_lng[:, 0], axis=1)
lng = gs.expand_dims(lat_lng[:, 1] + gs.pi, axis=1)
cities_from_dhoom = sphere.spherical_to_extrinsic(gs.concatenate([colat, lng], axis=1))

print(f'shape from JSON : {gs.shape(cities_from_json)}')
print(f'shape from DHOOM: {gs.shape(cities_from_dhoom)}')
print(f'arrays match    : {bool(gs.all(gs.isclose(cities_from_json, cities_from_dhoom)))}')
print(f'on the sphere   : {bool(gs.all(sphere.belongs(cities_from_dhoom)))}')
print(f'first city      : {names[0]} -> {cities_from_dhoom[0]}')

shape from JSON : (50, 3)
shape from DHOOM: (50, 3)
arrays match    : True
on the sphere   : True
first city      : Tokyo -> [ 0.61993792 -0.52479018  0.58332859]


## 2. Size comparison

DHOOM's whole point is to drop structural redundancy. Let's see what that looks like on a real geomstats dataset.

In [2]:
json_bytes = os.path.getsize(data_utils.CITIES_PATH)
dhoom_bytes = os.path.getsize(data_utils.CITIES_DHOOM_PATH)
reduction = (1 - dhoom_bytes / json_bytes) * 100

print(f'cities.json : {json_bytes:>6} bytes')
print(f'cities.dhoom: {dhoom_bytes:>6} bytes')
print(f'reduction   : {reduction:.1f}% smaller')

cities.json :  14083 bytes
cities.dhoom:   4665 bytes
reduction   : 66.9% smaller


## 3. What does the DHOOM file actually look like?

Unlike binary formats, DHOOM stays human-readable. The first line is the **fiber** — the schema header — and every subsequent line is a record. Field modifiers like `&` (string interning) and `|default` (modal defaults) appear in the header; records only carry the data that varies.

In [3]:
with open(data_utils.CITIES_DHOOM_PATH, encoding='utf-8') as f:
    contents = f.read()

lines = contents.splitlines()
print(f'(file has {len(lines)} lines total — showing first 6)')
print()
for line in lines[:6]:
    print(line)

(file has 52 lines total — showing first 6)

data{capital&, city, city_ascii, lat, lng, country, iso2, iso3, admin_name, population, id}:
&capital[primary, , admin, minor]
0, Tokyo, Tokyo, 35.685, 139.7514, Japan, JP, JPN, Tōkyō, 35676000, 1392685764
1, New York, New York, 40.6943, -73.9249, United States, US, USA, New York, 19354922, 1840034016
0, Mexico City, Mexico City, 19.4424, -99.131, Mexico, MX, MEX, Ciudad de México, 19028000, 1484247881
2, Mumbai, Mumbai, 19.017, 72.857, India, IN, IND, Mahārāshtra, 18978000, 1356226629


Reading top-to-bottom: the header `data{capital&, city, city_ascii, lat, lng, country, iso2, iso3, admin_name, population, id}:` declares the fiber. `capital&` means the `capital` field is **interned** — its values come from a pool declared on the next line. The pool line `&capital[primary, , admin, minor]` lists the four distinct values seen across all records, and each record uses a small integer index instead of the full string.

Geometrically: the bundle is `(F, E, B, π)` where `F` is the fiber (the 11-field schema), `B` is the index set (50 cities, ordered), and `E` is the total space of all city records. The DHOOM document is a serialized section of `E`.

## 4. Round-tripping with `save_dhoom`

`save_dhoom` and `load_dhoom` are inverses on JSON-shaped data. Quick demonstration on a tiny dataset:

In [4]:
import tempfile

original = [
    {'id': 1, 'name': 'alpha', 'value': 1.5},
    {'id': 2, 'name': 'beta', 'value': 2.5},
    {'id': 3, 'name': 'gamma', 'value': 3.5},
]

with tempfile.NamedTemporaryFile(suffix='.dhoom', mode='w', delete=False, encoding='utf-8') as tf:
    tmp_path = tf.name

data_utils.save_dhoom(original, tmp_path)
recovered = data_utils.load_dhoom(tmp_path)

with open(tmp_path, encoding='utf-8') as f:
    print('--- DHOOM representation ---')
    print(f.read())

print(f'recovered == original: {recovered == original}')

os.unlink(tmp_path)

--- DHOOM representation ---
data{id@1, value@1.5, name}:
alpha
beta
gamma

recovered == original: True


## Further reading

- [DHOOM format specification](https://dhoom.dev) — the full spec, including all field modifiers and the formal grammar.
- Davis, B. R. (2024). *The Geometry of Sameness*. Amazon KDP.
- Davis, B. R. (2026). *The Double Cover Principle*. Zenodo.